In [1]:
import calendar
import json
import os
import pickle
import random
import re
import sys
from datetime import date
from typing import List

import dateutil
import graphistry
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
import seaborn as sns
from pyspark.sql import DataFrame, SparkSession

In [2]:
GRAPHISTRY_KEY_ID = "91LX0MJCAF"   #os.environ["GRAPHISTRY_KEY_ID"]
GRAPHISTRY_API_KEY = "D76N4294TL52ELOB" #os.environ["GRAPHISTRY_API_KEY"]

In [3]:
# Configuration for Graphistry
GRAPHISTRY_PARAMS = {
    "play": 500,
    "pointOpacity": 0.7,
    "edgeOpacity": 0.3,
    "edgeCurvature": 0.3,
    "showArrows": True,
    "gravity": 0.15,
    "showPointsOfInterestLabel": False,
    "labels": {
        "shortenLabels": False,
    },
}
FAVICON_URL = "https://graphlet.ai/assets/icons/favicon.ico"
LOGO = {"url": "https://graphlet.ai/assets/Branding/Graphlet%20AI.svg", "dimensions": {"maxWidth": 100, "maxHeight": 100}}

In [4]:
# Initialize a SparkSession
spark: SparkSession = (
    SparkSession.builder.appName("Stack Overflow Pregel API")
    # Lets the Id:(Stack Overflow int) and id:(GraphFrames ULID) coexist
    .config("spark.sql.caseSensitive", True)
    .getOrCreate()
)
spark.sparkContext.setCheckpointDir("/tmp/graphframes-checkpoints")

25/05/28 14:06:51 WARN Utils: Your hostname, mikecabin.local resolves to a loopback address: 127.0.0.1; using 192.168.1.163 instead (on interface en0)
25/05/28 14:06:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/28 14:06:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Loading the Knowledge Graph

We used BAML and Gemini 2.5. Pro to extract the entities below and then PySpark to process the raw records into Parquet files and another script to create a node/edge list for the knowledge graph.

In [5]:
!ls ../data/refined_knowledge_graph/

companies.parquet                    nodes.txt
company_ticker_relationships.parquet nodes2.csv
edges.csv                            products.parquet
edges.parquet                        technologies.parquet
edges.tsv                            tickers.parquet
nodes.csv                            vertices.parquet


In [29]:
tickers = pd.read_parquet("../data/refined_knowledge_graph/tickers.parquet")

In [31]:
tickers

,exchange,name,symbol
0,None,ABLZF,None
1,KRX,Samsung Electronics,005930
2,JP,JSR,4185
3,JP,Tokyo Ohka Kogyo,4186
4,None,DISCO,6146
...,...,...,...
116,None,Verizon,VZ
117,None,Western Digital,WDC
118,None,Wells Fargo,WFC
119,None,Wolfspeed,WOLF


In [32]:
tickers["symbol"].unique()

array([None, '005930', '4185', '4186', '6146', '6756', '6857', '6871',
       '8035', '980', 'A', 'AAOI', 'AAPL', 'ACLS', 'ADBE', 'ADI', 'AEHR',
       'AIXA', 'AMAT', 'AMD', 'AMKR', 'AMZN', 'ANET', 'ANSS', 'APH',
       'ASMI', 'ASML', 'ASMPT', 'ASX', 'ATEYY', 'AVGO', 'AWE', 'BABA',
       'BIDU', 'CAMT', 'CDNS', 'COHU', 'CRDO', 'CRM', 'CRUS', 'CSCO',
       'DB', 'DELL', 'DISH', 'DLR', 'DOW', 'ENTG', 'EQIX', 'FDX', 'FLEX',
       'FORM', 'GE', 'GFS', 'GOOG', 'GOOGL', 'GSAT', 'HPE', 'HPQ', 'IBM',
       'IFX', 'ILMN', 'INTC', 'JBL', 'JNPR', 'KKR', 'KLA', 'KLAC', 'KLIC',
       'LITE', 'LMT', 'LRCX', 'MCHP', 'META', 'MKSI', 'MPWR', 'MRVL',
       'MSFT', 'MTSI', 'MU', 'NOK', 'NVDA', 'NVMI', 'NXPI', 'ON', 'ONTO',
       'ORCL', 'PACB', 'PDD', 'POWI', 'QCOM', 'QRVO', 'SIE', 'SMCI',
       'SMI', 'SMSN', 'SNAP', 'SNE', 'SNPS', 'SONY', 'SSNLF', 'STM',
       'STX', 'SWKS', 'TCEHY', 'TEL', 'TER', 'TPRO', 'TSLA', 'TSM',
       'TWTR', 'TXN', 'UBER', 'UBS', 'UMC', 'VICR', 'VMW', 'VZ', 'WDC',


In [33]:
vertices = pd.read_parquet("../data/refined_knowledge_graph/companies.parquet")

In [34]:
vertices[vertices["ticker"].isna() == False]

,ceo,description,employees,founded_year,headquarters_location,linkedin_url,name,revenue_usd,ticker,website_url
16,None,ASM International is a Dutch multinational cor...,NaN,NaN,None,None,ASM International,NaN,"{'exchange': None, 'name': 'ASM International'...",None
20,None,ASM Pacific Technology is a company that desig...,NaN,NaN,None,None,ASM Pacific Technology,NaN,"{'exchange': None, 'name': 'ASM Pacific Techno...",None
35,None,Acer Inc. is a Taiwanese multinational hardwar...,NaN,NaN,None,None,Acer Inc,NaN,"{'exchange': 'TT', 'name': 'Acer', 'symbol': '...",None
40,None,Adobe is a software company that develops soft...,NaN,NaN,None,None,Adobe Inc.,NaN,"{'exchange': None, 'name': 'Adobe', 'symbol': ...",None
43,None,AMD is a semiconductor company that develops c...,NaN,NaN,None,None,Advanced Micro Devices Inc.,NaN,"{'exchange': None, 'name': 'AMD', 'symbol': 'A...",None
...,...,...,...,...,...,...,...,...,...,...
1018,None,Vicor is a company that develops and manufactu...,NaN,NaN,None,None,Vicor Corporation,NaN,"{'exchange': None, 'name': 'Vicor', 'symbol': ...",None
1030,None,Wells Fargo & Company is an American multinati...,NaN,NaN,None,None,Wells Fargo & Company,NaN,"{'exchange': None, 'name': 'Wells Fargo', 'sym...",None
1031,None,Western Digital Corporation is an American com...,NaN,NaN,None,None,Western Digital,NaN,"{'exchange': None, 'name': 'Western Digital', ...",None
1042,None,"Wolfspeed, Inc. is an American company special...",NaN,NaN,None,None,"Wolfspeed, Inc.",NaN,"{'exchange': None, 'name': 'Wolfspeed', 'symbo...",None


In [21]:
vertices[vertices['entity_type'] == "product"]


,id,entity_type,properties
1075,Cisco Co-Packaged Optics,product,"{""company"":{""description"":""A technology compan..."
1076,CoreWeave Bare Metal,product,"{""company"":{""description"":""A GPU cloud provide..."
1077,H100,product,"{""company"":{""description"":""A leading GPU manuf..."
1078,IFS Packaging Units,product,"{""company"":{""ceo"":""Pat Gelsinger"",""description..."
1079,Kioxia highest density CMOS bonded to array (C...,product,"{""company"":{""description"":""A semiconductor com..."
...,...,...,...
2127,Cicero,product,"{""company"":{""description"":""A company that has ..."
2128,LiquidSecurity 2,product,"{""company"":{""description"":""A semiconductor com..."
2129,Qualcomm Boundless XR Demo,product,"{""company"":{""description"":""A technology compan..."
2130,Trainium 2,product,"{""company"":{""description"":""A major technology ..."


25/04/29 17:41:35 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 304777 ms exceeds timeout 120000 ms
25/04/29 17:41:35 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/29 17:41:39 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o